In [1]:
# Imports
import glob
import os
import pandas as pd
import numpy as np
import regex as re
import html
from datetime import datetime as dt
import json

from zipfile import ZipFile
import unicodedata

# Load Data (News Sites)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Preprocess

In [3]:
def preprocess_news_sites(source, text):
  result = html.unescape(html.unescape(str(text)))
  result = unicodedata.normalize('NFKC', result)
  result = result.replace('\xad', '')
  result = result.strip()

  if source == "abante":
    # HEADER
    result = re.sub(r"^[Nn][Ii]([Nn][Aa])?:?\s+.+([A-zÑñ]+)[ ]*\n+", "", result).strip() # author
    result = re.sub('^([A-Z][A-zÑñ]+\.?,?\s?)+\s*[-‒–—]+\s*(?=[A-Z"‘’“”])', "", result).strip() # location
    result = re.sub("^(([A-Z'\"‘“-]+-?[[:punct:]]?)+\s)+", lambda match: match.group(0)[0:match.group(0).find(next(filter(str.isalpha, match.group(0)))) + 1] + match.group(0)[match.group(0).find(next(filter(str.isalpha, match.group(0)))) + 1:].lower(), result).strip() # ALL CAPS letter at the start

    # FOOTER
    result = re.sub('\s*\([A-ZÑ][A-zÑñ,\/. -]+\)?$', "", result) # author names in parenthesis

  elif source == "abscbn":
    # HEADER
    result = re.sub("^\(UPDATED?\)\s*", "", result).strip() # (UPDATE)
    result = re.sub('^([A-Z][A-zÑñ]+\.?,?\s?)+\s*(\(UPDATED?\))?\s*[-‒–—]+\s*(?=[A-Z"‘’“”])', "", result) # location
    result = re.sub('^([A-Z][A-zÑñ]+\.?,?\s?)+\s*[-‒–—]+\s*(\(UPDATED?\))?\s*(?=[A-Z"‘’“”])', "", result) # location

    # FOOTER
    result = re.sub("\n[A-Z\s]+:?$", "", result).strip() # "RELATED VIDEO" etc
    result = re.sub("\n(\w+\s?)+:$", "", result).strip() # "Related videos:" etc
    result = re.sub('\s*Nagpa-Patrol,.+$', "", result).strip() # "Nagpa-patrol, ..."
    result = re.sub("(?<=[\n.])(\s[A-Z\s]+:?)?\s*[-‒–—]+[\wÑñ. ,-]+$", "", result).strip() # "RELATED VIDEO Ulat ni..." or "Ulat ni..."

  elif source == "balita":
    # HEADER \w
    result = re.sub(r"^[Nn][Ii]([Nn][Aa])?:?\s+.+([A-zÑñ]+)[ ]*\n+", "", result).strip() # author
    result = re.sub("^([A-Z][A-zÑñ]+\.?,?\s?){1,5}(\s+\([A-zÑñ]+\))?\s*[-‒–—]+\s*(?=[A-Z'\"‘’“”])", "", result).strip() # location
    result = re.sub("^(([A-Z'\"‘“-]+-?[[:punct:]]?)+\s)+", lambda match: match.group(0)[0:match.group(0).find(next(filter(str.isalpha, match.group(0)))) + 1] + match.group(0)[match.group(0).find(next(filter(str.isalpha, match.group(0)))) + 1:].lower(), result).strip() # ALL CAPS letter at the start

    # FOOTER
    result = re.sub('\s*[-‒–—\n[]\s*[A-ZÑ]((([A-zÑñ]+,?[ \/-]?)|([A-ZÑ][.][ ])))+$', "", result) # author names
    result = re.sub('\s+[A-ZÑ]((([A-zÑñ]+,?[ \/-]?)|([A-ZÑ][.][ ])))+$', "", result) # author names
    result = re.sub('\s*\([A-ZÑ][A-zÑñ,\/. -]+\)$', "", result) # author names in parenthesis

  elif source == "bandera":
    # HEADER
    result = re.sub("^[A-Z]{2,}[^a-z]+?\n", "", result).strip()
    result = re.sub("^([A-Z][A-zÑñ]+\.?,?\s?){1,5}(\s+\([A-zÑñ]+\))?\s*[-‒–—]+\s*(?=[A-Z'\"‘’“”])", "", result).strip()
    result = re.sub('^(["‘“]*[A-Z -]{2,}(?![a-z]))', lambda match: match.group(0)[0:match.group(0).find(next(filter(str.isalpha, match.group(0)))) + 1] + match.group(0)[match.group(0).find(next(filter(str.isalpha, match.group(0)))) + 1:].lower(), result).strip() # ALL CAPS letter at the start

  elif source == "gma":
    # HEADER
    result = re.sub('MANILA (-)?', "", result)
    # FOOTER
    result = re.sub('[-‒–—]+\s*([\/\wÑñ]+-?\.?,?\s*)+[A-z]$', "", result).strip()
    result = re.sub('\s*Click here.+:?\s*$', "", result)
    # result = re.sub('\s*Panoorin.*[[:punct:]]$', "", result)

  elif source == "mb":
    # HEADER
    result = re.sub('^[Bb][Yy]\s.+\n', "", result).strip()
    # result = re.sub("^([A-Z][A-zÑñ]+,?\s?){1,5}(\s+\([A-zÑñ]+\))?\s*[-‒–—]+\s*(?=[A-Z'\"‘’“”])", "", result).strip() # location
    result = re.sub("^([A-Z][A-zÑñ]*\.?,?\s?){1,7}(\s+\([A-zÑñ]+\))?\s*[-‒–—]+\s*(?=[A-Z'\"‘’“”])", "", result).strip() # location

    # MID
    result = re.sub('\nREAD MORE:.*(?=\n|$)', "", result, flags=re.IGNORECASE).strip()
    result = re.sub('\nRELATED STORY:.*(?=\n|$)', "", result, flags=re.IGNORECASE).strip()

    # FOOTER
    result = re.sub('(?<=[[:punct:]])\s*[(\[](\w+\.?,?\s?)+[)\]]?$', "", result).strip() # author

  elif source == "philstar":
    # HEADER
    result = re.sub('^Dear Dr.? ?Love,?\s*', "", result, flags=re.IGNORECASE)

    # FOOTER
    result = re.sub('\s* Dr.? ?Love\s*$', "", result, flags=re.IGNORECASE)

  elif source == "radyoinquirer":
    # HEADER
    result = re.sub('^\((\S+\s)*?\S+\)', "", result)
  
  return result.strip()

def preprocess(text):
    # html
    result = html.unescape(html.unescape(str(text)))
    result = unicodedata.normalize('NFKC', result)

    # Email
    result = re.sub(r"[\SÑñ]+@([\SÑñ]+\.)+[\SÑñ]+", " XX_EMAIL ", result)

    # urls
    result = re.sub(r"https?:\/\/([\w\-_]+\.)+([\w\-_]+)+(\/[^\s]+)*", " XX_URL ", result)
    result = re.sub(r"([\w\-_]+\.)+(com|net|org|co|us|ph|io)(\/[^\s]+)*", " XX_URL ", result, flags=re.IGNORECASE)
    
    # twitter username mentions
    result = re.sub(r"(?<!\S)@[^\s.,!?]+(?!\S)", " XX_USERNAME ", result)

    # extra spaces
    result = re.sub(r'\s\s+',' ', result) # replace 2++ white space with 1 white space

    # source specific artifacts removal goes here
    
    return result.strip()

In [4]:
col_name = 'full_article'

data_dir = "Raw Data"
new_data_dir = "RAW_COHFIE"
source = "news_sites"

#news_sites_with_categories = ["abante", "abscbn", "balita", "bandera", "mb", "philstar", "radyoinquirer"]
news_sites_with_categories = ["balita", "bandera", "mb", "radyoinquirer"]
#news_sites_wo_categories = ["gma"]

In [5]:
dir = "/content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection"

# Playground (skip)

In [ ]:
# year = 2017
# month = 3

# df = pd.DataFrame()
# if source_type in news_sites_with_categories:
#   for category in os.listdir(f'{dir}/{data_dir}/{source}/{source_type}'):
#     print(f'{dir}/{data_dir}/{source}/{source_type}/{category}')
#     if os.path.exists(f'{dir}/{data_dir}/{source}/{source_type}/{category}/{year}/{month}/data.zip'):
#         zipfile = ZipFile(f'{dir}/{data_dir}/{source}/{source_type}/{category}/{year}/{month}/data.zip', 'r')

#         for file in zipfile.namelist():
#             newData = pd.read_json(zipfile.open(file), orient='records', lines=True)
#             df = df.append(newData, ignore_index=True)
                        
# elif source_type in news_sites_wo_categories:
#   if os.path.exists(f'{dir}/{data_dir}/{source}/{source_type}/{year}/{month}/data.zip'):
#       zipfile = ZipFile(f'{dir}/{data_dir}/{source}/{source_type}/{year}/{month}/data.zip', 'r')

#       for file in zipfile.namelist():
#           newData = pd.read_json(zipfile.open(file), orient='records', lines=True)
#           df = df.append(newData, ignore_index=True)

In [ ]:
# import random

In [ ]:
# df = pd.DataFrame()
# for year in range(2000, 2022):
#     # random.seed(42)
#     for month in random.sample(range(1, 13), 3):
#     # for month in range(1, 13):
        
#         if source_type in news_sites_with_categories:
#             for category in os.listdir(f'{dir}/{data_dir}/{source}/{source_type}'):
#                 if os.path.exists(f'{dir}/{data_dir}/{source}/{source_type}/{category}/{year}/{month}/data.zip'):
#                     print(f'{data_dir}/{source}/{source_type}/{category}/{year}/{month}/data.zip')
#                     zipfile = ZipFile(f'{dir}/{data_dir}/{source}/{source_type}/{category}/{year}/{month}/data.zip', 'r')

#                     for file in zipfile.namelist():
#                         newData = pd.read_json(zipfile.open(file), orient='records', lines=True)
#                         df = df.append(newData, ignore_index=True)
                        
#         elif source_type in news_sites_wo_categories:
#             if os.path.exists(f'{dir}/{data_dir}/{source}/{source_type}/{year}/{month}/data.zip'):
#                 # print(f'{dir}/{data_dir}/{source}/{source_type}/{year}/{month}/data.zip')
#                 zipfile = ZipFile(f'{dir}/{data_dir}/{source}/{source_type}/{year}/{month}/data.zip', 'r')

#                 for file in zipfile.namelist():
#                     newData = pd.read_json(zipfile.open(file), orient='records', lines=True)
#                     df = df.append(newData, ignore_index=True)

In [ ]:
# len(df)

In [ ]:
# random_state = 50

# df_sample = df.sample(n=50, random_state=random_state)
# df_sample = df
# df_sample = df[df['full_article'].apply(lambda article: "UPDATED" in article)]

In [ ]:
# df_sample['full_article_2'] = df_sample['full_article'].apply(lambda article: preprocess(preprocess_news_sites(source_type, article)))

In [ ]:
# df_sample['full_article_repr'] = df_sample['full_article'].apply(lambda article: repr(article))
# df_sample['full_article_2_repr'] = df_sample['full_article_2'].apply(lambda article: repr(article))

# df_sample['full_article_start'] = df_sample['full_article'].apply(lambda article: repr(article[:70]))
# df_sample['full_article_2_start'] = df_sample['full_article_2'].apply(lambda article: repr(article[:70]))

# df_sample['full_article_end'] = df_sample['full_article'].apply(lambda article: repr(article[-70:]))
# df_sample['full_article_2_end'] = df_sample['full_article_2'].apply(lambda article: repr(article[-70:]))

In [ ]:
# display(df_sample.loc[58][['full_article', 'full_article_2']])

# df_sample[['full_article_repr', 'full_article_2_repr']].style

In [ ]:
# display(df_sample[['full_article_start', 'full_article_2_start']].style)
# display(df_sample[['full_article_end', 'full_article_2_end']].style)
# display(df_sample[['full_article_repr', 'full_article_2_repr']].style)

In [ ]:
# df[df['full_article'].apply(lambda article: "GMANews.TV" in article)][['full_article', 'full_article_2']].head(50)

In [ ]:
# df[['full_article', 'full_article_2']].style

In [ ]:
# print(repr(df['full_article_2'].iloc[269]))

In [ ]:
# df[['full_article', 'full_article_2']].style

---

# Main

In [20]:
source_type = "radyoinquirer"

In [21]:
years = []
if source_type in news_sites_with_categories:
  for cat in os.listdir(f'{dir}/{data_dir}/{source}/{source_type}'):
    for year in os.listdir(f'{dir}/{data_dir}/{source}/{source_type}/{cat}'):
      years.append(year)
elif source_type in news_sites_wo_categories:
  for year in os.listdir(f'{dir}/{data_dir}/{source}/{source_type}'):
      years.append(year)

years = sorted(set(years))

years = [2022] # change
months = [1, 2] # change

for year in years:
    for month in months:
        df = pd.DataFrame()
        
        if source_type in news_sites_with_categories:
            for category in os.listdir(f'{dir}/{data_dir}/{source}/{source_type}'):
                if os.path.exists(f'{dir}/{data_dir}/{source}/{source_type}/{category}/{year}/{month}/data.zip'):
                    # print(f'{data_dir}/{source}/{source_type}/{category}/{year}/{month}/data.zip')
                    zipfile = ZipFile(f'{dir}/{data_dir}/{source}/{source_type}/{category}/{year}/{month}/data.zip', 'r')

                    for file in zipfile.namelist():
                        newData = pd.read_json(zipfile.open(file), orient='records', lines=True)
                        df = df.append(newData, ignore_index=True)
                        
        elif source_type in news_sites_wo_categories:
            if os.path.exists(f'{dir}/{data_dir}/{source}/{source_type}/{year}/{month}/data.zip'):
                # print(f'{dir}/{data_dir}/{source}/{source_type}/{year}/{month}/data.zip')
                zipfile = ZipFile(f'{dir}/{data_dir}/{source}/{source_type}/{year}/{month}/data.zip', 'r')

                for file in zipfile.namelist():
                    newData = pd.read_json(zipfile.open(file), orient='records', lines=True)
                    df = df.append(newData, ignore_index=True)

        if not df.empty:
            df[col_name] = df[col_name].apply(lambda text: preprocess(preprocess_news_sites(source_type, text)))

            if 'category' not in df.columns:
                df['category'] = np.nan

            if 'tags' not in df.columns:
                df['tags'] = pd.Series([[] for _ in range(len(df.index))])
            else:
                df['tags'] = df['tags'].apply(lambda tags: [] if tags == "" else tags)

            df.rename(columns = {'full_article':'text'}, inplace = True)
            df_res = df[['text', 'category', 'date', 'tags', 'url']]

            output_path = f"{dir}/{new_data_dir}/{source}/{source_type}/{year}/{month}"

            if not os.path.exists(f'{output_path}'):
                os.makedirs(f'{output_path}')

            df_res.to_json(path_or_buf=f'{output_path}/{month}.json', orient="records")
            print("\n\n\nDone: " + f'{output_path}/{month}.json')
            display(df_res.head())
        else:
            print("Skipped: " + f'{source}/{source_type}/{year}/{month}')




Done: /content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/RAW_COHFIE/news_sites/radyoinquirer/2022/1/1.json


,text,category,date,tags,url
0,Pansamantalang isinara ang Quiapo Church mula ...,metro,2022-01-03 14:32:00+08:00,[],https://radyo.inquirer.net/306750/quiapo-churc...
1,"Sa pagpasok ng taong 2022, hinikayat ni Quezon...",metro,2022-01-03 14:58:00+08:00,[],https://radyo.inquirer.net/306759/mayor-belmon...
2,Iniimbestigahan na ng awtoridad ang pagkamatay...,metro,2022-01-03 19:27:00+08:00,[],https://radyo.inquirer.net/306796/suicide-hini...
3,Tumatanggap na muli ang Justice Jose Abad Sant...,metro,2022-01-03 19:37:00+08:00,[],https://radyo.inquirer.net/306800/justice-jose...
4,Ititigil na muna ni Aksyon Demokratiko preside...,metro,2022-01-04 10:32:00+08:00,[],https://radyo.inquirer.net/306856/listening-to...





Done: /content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/RAW_COHFIE/news_sites/radyoinquirer/2022/2/2.json


,text,category,date,tags,url
0,Magpapatuloy na ang elimination round ng PBA G...,sports,2022-02-07 15:54:00+08:00,[],https://radyo.inquirer.net/308873/pba-governor...
1,Tatlong manlalaro pa ng South Korea ang nag-po...,sports,2022-02-22 21:03:00+08:00,[],https://radyo.inquirer.net/309782/south-korea-...
2,Pinapurihan ni Senator Cynthia Villar ang nag-...,sports,2022-02-27 20:52:00+08:00,[],https://radyo.inquirer.net/310035/game-fowl-in...
3,Hindi naging sapat ang pagpupursige ng Gilas P...,sports,2022-02-27 21:13:00+08:00,[],https://radyo.inquirer.net/310038/gilas-pilipi...
4,Magsasagawa ang Quezon City government ng swab...,metro,2022-02-02 09:39:00+08:00,[],https://radyo.inquirer.net/308574/qc-lgu-magsa...


---

# Check afterwards

In [ ]:
print(f'{dir}/{source}/{source_type}')

/content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/news_sites/mb


In [ ]:
for year in f'{dir}/{data_dir}/{source}/{source_type}'

In [ ]:
source_type = news_sites_with_categories[3]
print(source_type)

NameError: ignored

In [ ]:
test_df = pd.read_json(f"{dir}/{new_data_dir}/{source}/{source_type}/2021/5/5.json")

In [ ]:
test_df

,text,category,date,tags,url
0,The easterlies or the warm wind originating fr...,national,2021-04-30 23:07:00,"[EASTERLIES, PAGASA, weather]",https://mb.com.ph/2021/05/01/generally-fair-we...
1,Hopes are high that 20 million doses of the Ru...,national,2021-04-30 23:30:00,[],https://mb.com.ph/2021/05/01/mb-daily-news-upd...
2,While most Filipinos are longing to bust out o...,national,2021-04-30 23:51:00,"[DOT, Intramuros, labor day, Siquijor, tourism]",https://mb.com.ph/2021/05/01/as-tourists-wait-...
3,Preparations are underway for the possible bea...,national,2021-05-01 00:23:00,[],https://mb.com.ph/2021/05/01/claretian-priest-...
4,Senate Minority Leader Franklin Drilon on Sund...,national,2021-05-01 00:43:00,"[ASEAN, china, diplomatic protests, Franklin D...",https://mb.com.ph/2021/05/01/drilon-to-govt-dr...
...,...,...,...,...,...
2525,"Metro Manila, Bulacan, Cavite, Laguna, and Riz...",national,2021-05-31 15:08:00,"[Community Quarantine, COVID-19, DUTERTE, GCQ,...",https://mb.com.ph/2021/05/31/one-more-month-of...
2526,The Philippine Science High School (PSHS) Syst...,national,2021-05-31 15:26:00,"[ADMISSION OFFERS, PISAY, PSHS]",https://mb.com.ph/2021/05/31/six-more-pisay-sc...
2527,The Philippines will keep its borders closed t...,national,2021-05-31 15:36:00,"[extend, extension, india, June 15, Philippine...",https://mb.com.ph/2021/05/31/ph-extends-travel...
2528,The Department of Science and Technology (DOST...,national,2021-05-31 15:38:00,"[CMU, dost, SPRAY DRYING TECHNOLOGY]",https://mb.com.ph/2021/05/31/cmu-study-on-spra...
